In [1]:
# Cell 1: Imports & Setup
from pathlib import Path
import json
import numpy as np
import pandas as pd
import polars as pl
import gc

data_base_dir = Path('data/regionwise_yearwise_samples')
output_dir = Path('data/feature_selection_artifacts')
output_dir.mkdir(parents=True, exist_ok=True)

TARGET_REGION = 'DES'
REGION_DIR = data_base_dir / TARGET_REGION

cols_to_drop = [
    'class', 'system:index', 'area_m2', 'area_proportion', 'usable_count',
    '.geo', 'q1_count', 'q2_count', 'q3_count', 'q4_count', 'quarters_present',
    'geometry', 'ndvi_q1_median', 'ndvi_q2_median', 'ndvi_q3_median',
    'ndvi_q4_median', 'tir_count', 'tir_mad', 'tir_median', 'snow_count',
    'year'
]

old_codes = [5, 3, 4, 12, 11, 66, 19, 36, 9, 41, 24, 23, 75, 30, 25, 33, 76, 34, 61]
new_codes = [5, 3, 4, 12, 11, 66, 19, 14, 14, 14, 22, 22, 22, 22, 22, 33, 76, 34, 61]
class_remap_dict = dict(zip(old_codes, new_codes))

In [2]:
# Cell 2: Sample 50k Rows for Fast Correlation
lazy_ds = pl.scan_parquet(REGION_DIR / "**/*.parquet")
schema = lazy_ds.collect_schema()
feature_cols = [c for c in schema.names() if c not in cols_to_drop]

df_sample = (
    lazy_ds
    .with_columns(
        pl.col('class').replace_strict(class_remap_dict, default=pl.col('class')).cast(pl.Int32)
    )
    .select(feature_cols + ['class'])
    .collect()
    .sample(n=min(50000, 4500000), seed=42)
)

X_sample = df_sample.select(feature_cols).to_numpy().astype(np.float32)
y_sample = df_sample['class'].to_numpy().astype(np.float32)

del df_sample
gc.collect()

print(f"Correlation sample ready: {X_sample.shape[0]:,} rows across {len(feature_cols)} features.")

Correlation sample ready: 50,000 rows across 104 features.


In [3]:
# Cell 3: Compute Pairwise & Target Correlation Matrices
# 1. Feature-to-Feature Pearson correlation
corr_matrix = np.abs(np.corrcoef(X_sample, rowvar=False))
corr_df = pd.DataFrame(corr_matrix, index=feature_cols, columns=feature_cols)

# 2. Vectorized Feature-to-Target correlation (|r(X, y)|)
y_norm = (y_sample - np.mean(y_sample)) / np.std(y_sample)
X_norm = (X_sample - np.mean(X_sample, axis=0)) / np.std(X_sample, axis=0)
target_corr = np.abs(np.mean(X_norm * y_norm[:, np.newaxis], axis=0))
target_corr_series = pd.Series(target_corr, index=feature_cols)

/home/nirzaree/Code/GitClones/IOLN/.IOLN/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3036: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/nirzaree/Code/GitClones/IOLN/.IOLN/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3037: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/tmp/ipykernel_47483/694834287.py:8: RuntimeWarning: invalid value encountered in divide
  X_norm = (X_sample - np.mean(X_sample, axis=0)) / np.std(X_sample, axis=0)


In [4]:
# Cell 4: Drop Collinear Features (|r| > 0.80) & Export Artifact
threshold = 0.80
upper = corr_df.where(np.triu(np.ones(corr_df.shape), k=1).astype(bool))

to_drop = set()
for col in upper.columns:
    correlated_features = upper.index[upper[col] > threshold].tolist()
    if correlated_features:
        for feat in correlated_features:
            if target_corr_series[feat] < target_corr_series[col]:
                to_drop.add(feat)
            else:
                to_drop.add(col)

uncorrelated_features = [col for col in feature_cols if col not in to_drop]

output_artifact = {
    "region": TARGET_REGION,
    "corr_threshold": threshold,
    "total_raw_features": len(feature_cols),
    "uncorrelated_features": uncorrelated_features,
    "dropped_features": sorted(list(to_drop))
}

output_path = output_dir / f"{TARGET_REGION}_uncorrelated_features.json"
with open(output_path, "w") as f:
    json.dump(output_artifact, f, indent=2)

print(f"Initial raw features: {len(feature_cols)}")
print(f"Collinear features dropped (|r| > {threshold}): {len(to_drop)}")
print(f"Retained uncorrelated features: {len(uncorrelated_features)}")
print(f"Artifact saved to -> {output_path}")

Initial raw features: 104
Collinear features dropped (|r| > 0.8): 68
Retained uncorrelated features: 36
Artifact saved to -> data/feature_selection_artifacts/DES_uncorrelated_features.json
